# Vertical Velocity to sigmadot conversions


In [36]:
R_d = 287.05   # J kg^-1 K^-1
c_p = 1004.0   # J kg^-1 K^-1
kappa = R_d / c_p
g = 9.81       # m s^-2
p0 = 100000.0  # Pa

def exner(p):
    """
    Exner function with reference pressure of 1000 hPa
    :param p: pressure in hPa
    :return: Exner function value in J/(kg*K)
    """
    rcp = R_d / c_p
    psurf = p0 / 100.0  # convert reference pressure to hPa
    return c_p * (p / psurf) ** rcp


## SNAP implementation

Currently commented as "a very simple, probably too simple (!!!) conversion from m/s to model etadot/sigmadot "vertical velocity""

In [37]:
table_str = """
    66    10.00  0.00000  0.00987     10.    0.
    65    20.00  0.00000  0.01974     20.   10.
    64    40.01  0.00000  0.03948     40.   20.
    63    59.79  0.00024  0.05925     60.   20.
    62    78.64  0.00143  0.07905     80.   20.
    61    95.87  0.00431  0.09892    100.   20.
    60   111.24  0.00911  0.11889    120.   20.
    59   124.77  0.01584  0.13899    141.   20.
    58   136.50  0.02453  0.15925    161.   20.
    57   146.45  0.03518  0.17971    182.   21.
    56   154.65  0.04781  0.20043    202.   21.
    55   161.12  0.06247  0.22149    224.   21.
    54   165.90  0.07919  0.24292    245.   21.
    53   168.99  0.09793  0.26471    267.   22.
    52   170.43  0.11862  0.28681    289.   22.
    51   170.28  0.14114  0.30919    311.   22.
    50   168.64  0.16536  0.33179    334.   23.
    49   165.61  0.19113  0.35458    357.   23.
    48   161.34  0.21827  0.37750    380.   23.
    47   155.97  0.24659  0.40052    403.   23.
    46   149.66  0.27589  0.42359    426.   23.
    45   142.56  0.30597  0.44666    449.   23.
    44   134.84  0.33662  0.46970    471.   23.
    43   126.66  0.36765  0.49266    494.   23.
    42   118.17  0.39886  0.51549    517.   23.
    41   109.52  0.43007  0.53816    540.   23.
    40   100.82  0.46110  0.56061    562.   22.
    39    92.21  0.49179  0.58280    584.   22.
    38    83.78  0.52200  0.60469    606.   22.
    37    75.63  0.55160  0.62624    627.   21.
    36    67.82  0.58047  0.64739    648.   21.
    35    60.41  0.60850  0.66812    669.   21.
    34    53.46  0.63560  0.68837    689.   20.
    33    46.99  0.66171  0.70809    709.   20.
    32    41.03  0.68676  0.72725    728.   19.
    31    35.58  0.71069  0.74580    746.   18.
    30    30.64  0.73346  0.76370    764.   18.
    29    26.21  0.75503  0.78091    781.   17.
    28    22.27  0.77539  0.79737    798.   16.
    27    18.80  0.79449  0.81304    813.   16.
    26    15.77  0.81233  0.82789    828.   15.
    25    13.15  0.82890  0.84188    842.   14.
    24    10.90  0.84426  0.85502    855.   13.
    23     8.98  0.85849  0.86735    867.   12.
    22     7.34  0.87167  0.87892    879.   12.
    21     5.96  0.88386  0.88974    890.   11.
    20     4.79  0.89512  0.89985    900.   10.
    19     3.82  0.90552  0.90929    909.    9.
    18     3.02  0.91510  0.91807    918.    9.
    17     2.35  0.92392  0.92624    926.    8.
    16     1.81  0.93203  0.93382    934.    8.
    15     1.37  0.93949  0.94084    941.    7.
    14     1.02  0.94633  0.94734    947.    6.
    13     0.74  0.95261  0.95335    953.    6.
    12     0.52  0.95838  0.95889    959.    6.
    11     0.36  0.96367  0.96402    964.    5.
    10     0.23  0.96852  0.96875    969.    5.
     9     0.14  0.97299  0.97313    973.    4.
     8     0.07  0.97711  0.97719    977.    4.
     7     0.03  0.98093  0.98096    981.    4.
     6     0.01  0.98449  0.98450    984.    4.
     5     0.00  0.98783  0.98783    988.    3.
     4     0.00  0.99100  0.99100    991.    3.
     3     0.00  0.99405  0.99405    994.    3.
     2     0.00  0.99704  0.99704    997.    3.
     1     0.00  1.00000  1.00000   1000.    3.
"""
def parse_ahalf_bhalf_table(table_str):
    """
    Parses the given table string and returns a list of tuples containing the values.
    Each tuple corresponds to a row in the table and contains the values as floats.
    """
    lines = table_str.strip().split('\n')
    data = []
    for line in lines:
        values = line.split()
        if len(values) == 6:  # Ensure there are exactly 6 columns
            data.append(tuple(map(float, values)))
    # resize ahalf and bhalf to max level
    max_level = int(max(row[0] for row in data))
    ahalf = [0.0] * (max_level + 1)
    bhalf = [0.0] * (max_level + 1)
    vhalf = [0.0] * (max_level + 1)
    for row in data:
        ahalf[int(row[0])] = row[1]
        bhalf[int(row[0])] = row[2]
        vhalf[int(row[0])] = row[3]
    # typical values for theta (potential temperature) in these levels
    th = [0.0] * (max_level + 1)
    for row in data:
        pressure = row[4]  # pressure in hPa
        # Assuming a reference temperature of 300 K for potential temperature calculation
        reference_temperature = 300.0  # in Kelvin
        # Calculate potential temperature using the formula: theta = T * (p0 / p)^(R/cp)
        R = 287.05  # specific gas constant for dry air
        cp = 1004.0  # specific heat of dry air at constant pressure in J/(kg*K)
        p0 = 1000.0  # reference pressure in hPa
        theta = reference_temperature * (p0 / pressure) ** (R / cp)
        th[int(row[0])] = theta
    return ahalf, bhalf, vhalf, th, max_level
ahalf, bhalf, vhalf, th, nk = parse_ahalf_bhalf_table(table_str)


def snap2_5_17_gravity_to_sigmadot(vgrav: float, theta, ilayer: int):
    """
    !..vgrav ... a very simple, probably too simple (!!!) conversion
    !............. from m/s to model etadot/sigmadot "vertical velocity"
    :vgrav: vgrav=terminal velocity in m/s
    :theta: potential temperature in K
    :ilayer: model layer index (1-based)
    :return: vertical velocity in model etadot/sigmadot units in 1/s
    """
    ps = 1013.25 # surface pressure in hPa
    ginv = 1.0 / 9.81 # inverse of gravity in s^2/m


    k1 = ilayer
    if (k1 == nk):
        k1 = k1-1
    k2 = k1 + 1
    p = ahalf[k1] + bhalf[k1]*ps
    pi1 = exner(p)
    p = ahalf[k2] + bhalf[k2]*ps
    pi2 = exner(p)
    dz = th[k1]*(pi1-pi2)*ginv
    deta = vhalf[k1]-vhalf[k2]
    wg = vgrav*deta/dz
    return wg



## Hydrostatic formulas for converting from m/s to omega and sigmadot

In [38]:
def rho_from_theta_p(theta_k, p_pa, qv=None, ql=0.0, qi=0.0):
    """
    Compute density from potential temperature and pressure.
    If humidity is known, use virtual potential temperature.
    :param theta_k: potential temperature in K
    :param p_pa: pressure in Pa
    :param qv: specific humidity (kg/kg), optional
    :param ql: liquid water content (kg/kg), optional
    :param qi: ice content (kg/kg), optional
    :return: density in kg/m^3
    """
    if qv is None:
        theta_v = theta_k
    else:
        theta_v = theta_k * (1.0 + 0.61*qv - ql - qi)  # virtual potential temperature
    pi = (p_pa / p0) ** kappa
    T_v = theta_v * pi
    rho = p_pa / (R_d * T_v)
    return rho

def omega_from_w_theta(w_ms, theta_k, p_pa, qv=None, ql=0.0, qi=0.0):
    """
    Convert vertical velocity w (m/s) to omega (Pa/s) using theta and p.
    :param w_ms: vertical velocity in m/s
    :param theta_k: potential temperature in K
    :param p_pa: pressure in Pa
    :param qv: specific humidity (kg/kg), optional
    :param ql: liquid water content (kg/kg), optional
    :param qi: ice content (kg/kg), optional
    :return: omega in Pa/s
    """
    rho = rho_from_theta_p(theta_k, p_pa, qv=qv, ql=ql, qi=qi)
    return -rho * g * w_ms  # Pa/s

def sigma_dot_sigma_ps(omega_pas, p_pa, ps_pa, ps_tendency_pas=None):
    """
    Sigma = p/ps. Compute sigma-dot (1/s) from omega (Pa/s).
    Optionally include ps_tendency = Dps/Dt (Pa/s).
    """
    sigma = p_pa / ps_pa
    if ps_tendency_pas is None:
        return omega_pas / ps_pa
    else:
        return (omega_pas - sigma * ps_tendency_pas) / ps_pa

def sigma_dot_from_w_theta(w_ms, theta_k, p_pa, ps_pa, ps_tendency_pas=None, qv=None, ql=0.0, qi=0.0):
    """
    Sigma = p/ps. Compute sigma-dot (1/s) from w using theta and p.
    """
    omega = omega_from_w_theta(w_ms, theta_k, p_pa, qv=qv, ql=ql, qi=qi)
    return sigma_dot_sigma_ps(omega, p_pa, ps_pa, ps_tendency_pas)

def sigma_dot_general(omega_pas, p_pa, ps_pa, pt_pa=0.0, ps_tendency_pas=None, pt_tendency_pas=0.0):
    """
    General sigma: sigma = (p - pt)/(ps - pt), assuming pt is fixed unless pt_tendency_pas provided.
    Returns sigma-dot (1/s).
    """
    sigma = (p_pa - pt_pa) / (ps_pa - pt_pa)
    num = omega_pas - sigma * (ps_tendency_pas if ps_tendency_pas is not None else 0.0) \
          + (sigma - 1.0) * pt_tendency_pas
    den = (ps_pa - pt_pa)
    return num / den

def sigma_dot_from_w_theta_general(w_ms, theta_k, p_pa, ps_pa, pt_pa=0.0, ps_tendency_pas=None, pt_tendency_pas=0.0, qv=None, ql=0.0, qi=0.0):
    """
    General sigma: sigma = (p - pt)/(ps - pt), assuming pt is fixed unless pt_tendency_pas provided.
    Compute sigma-dot (1/s) from w using theta and p.
    """
    omega = omega_from_w_theta(w_ms, theta_k, p_pa, qv=qv, ql=ql, qi=qi)
    return sigma_dot_general(omega, p_pa, ps_pa, pt_pa, ps_tendency_pas, pt_tendency_pas)


In [39]:
import pandas as pd

rows = []
for i in range(1, 67):
    p_hpa = ahalf[i] + bhalf[i] * 1013.25
    snap_sigma_dot_s = snap2_5_17_gravity_to_sigmadot(1, th[i], i)
    general_sigma_dot_s = sigma_dot_from_w_theta_general(-1, th[i], p_hpa * 100.0, 1013.25 * 100.0, pt_pa=100.0)
    tendency_sigma_dot = sigma_dot_from_w_theta_general(-1,  th[i], p_hpa * 100.0, 1013.25 * 100.0, pt_pa=100.0, ps_tendency_pas=-0.1) # North Atlantic storm
    # some typical values for qv at different pressure levels
    qv = 0.005 * (p_hpa / 1000)**3 # very rough estimate of specific humidity
    sigma_dot_with_qv = sigma_dot_from_w_theta_general(-1, th[i], p_hpa * 100.0, 1013.25 * 100.0, pt_pa=100.0, qv=qv)
    rows.append({
        "level": i,
        "pressure_hpa": p_hpa,
        "theta_k": th[i],
        "snap_sigma_dot_s-1": snap_sigma_dot_s,
        "general_sigma_dot_s-1": general_sigma_dot_s,
        "vs_snap [%]": int(100* (general_sigma_dot_s - snap_sigma_dot_s) / snap_sigma_dot_s),
        "pressure_tendency_sigma_dot": tendency_sigma_dot,
        "vs_snap_tendency_1Pa/s [%]": int(100* (tendency_sigma_dot - snap_sigma_dot_s) / snap_sigma_dot_s),
        "qv [kg/kg]": qv,
        "sigma_dot_qv": sigma_dot_with_qv,
        "vs_snap_qv [%]": int(100* (sigma_dot_with_qv - snap_sigma_dot_s) / snap_sigma_dot_s)

    })

df = pd.DataFrame(rows)
# show the whole table
pd.set_option('display.max_rows', 70)
df


,level,pressure_hpa,theta_k,snap_sigma_dot_s-1,general_sigma_dot_s-1,vs_snap [%],pressure_tendency_sigma_dot,vs_snap_tendency_1Pa/s [%],qv [kg/kg],sigma_dot_qv,vs_snap_qv [%]
0,1,1013.250000,300.000000,0.000113,0.000114,0,0.000115,1,5.201395e-03,0.000113,0
1,2,1010.250780,300.257813,0.000113,0.000113,0,0.000114,1,5.155343e-03,0.000113,0
2,3,1007.221162,300.516626,0.000113,0.000113,0,0.000114,1,5.109101e-03,0.000113,0
3,4,1004.130750,300.776445,0.000112,0.000113,0,0.000114,1,5.062218e-03,0.000112,0
4,5,1000.918747,301.037277,0.000112,0.000112,0,0.000113,1,5.013794e-03,0.000112,0
5,6,997.544492,301.386643,0.000112,0.000112,0,0.000113,1,4.963258e-03,0.000111,0
6,7,993.957322,301.649867,0.000111,0.000111,0,0.000112,1,4.909906e-03,0.000111,0
7,8,990.126708,302.002448,0.000111,0.000111,0,0.000112,0,4.853358e-03,0.000111,0
8,9,986.022118,302.356890,0.000110,0.000111,0,0.000112,1,4.793249e-03,0.000110,0
9,10,981.582890,302.713210,0.000110,0.000110,0,0.000111,0,4.728800e-03,0.000110,0
